# Week 4 Day 2: First Supervised Models & Preprocessing

## Task 1: Preprocessing Plan & Implementation
List of numeric and categorical features, and the Scikit-Learn `ColumnTransformer` pipeline for solid preprocessing without data leakage.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load data and clean
print("Fetching dataset...")
data = fetch_openml('adult', version=2, as_frame=True)
df = data.frame

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert target to integer and drop 'fnlwgt' (survey weight) and 'education' string (redundant)
df['class'] = df['class'].apply(lambda x: 1 if x == '>50K' else 0).astype(int)
df = df.drop(columns=['fnlwgt', 'education'])

# 2. Reproducible Split (same as Day 1)
X = df.drop(columns=['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


In [ ]:
# 3. Explicitly list numeric and categorical features
numeric_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

# 4. Build the sklearn ColumnTransformer pipeline

# Numeric Pipeline:
# - SimpleImputer(strategy='median'): We use median because some numeric features (like capital-gain) are highly skewed. 
#   Mean imputation would be heavily influenced by extreme outliers, whereas median is robust.
#   (Alternative skipped: KNNImputer - skipped because it is much slower to train on large datasets).
# - StandardScaler: standardizes features to zero mean and unit variance, helping models that are sensitive to scale (like Logistic Regression/SVMs).
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline:
# - SimpleImputer(strategy='most_frequent'): A safe default for filling categorical gaps (like missing occupation).
#   (Alternative skipped: filling with a constant "Missing" category - skipped because we want the model to group 
#   missing rows with the dominant category to reduce sparsity, but "Missing" is also a valid alternative).
# - OneHotEncoder(handle_unknown='ignore'): Best for nominal categorical data. Using handle_unknown='ignore' prevents 
#   errors during prediction if a rare native-country shows up in the test set that wasn't in the train set.
#   (Alternative skipped: OrdinalEncoder - skipped because these are mostly nominal features without an inherent order, 
#   so forcing an ordinal ranking would confuse linear models).
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing Pipeline Built Successfully!")
preprocessor


## Task 2: Train Two Supervised Models (in Pipelines)
Here we build two complete pipelines: one combining our preprocessor with a Logistic Regression model, and the other with a Decision Tree Classifier. We then fit both pipelines exclusively on the training data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# 1. Logistic Regression Pipeline
# We use solver='lbfgs' (the default) which supports L2 regularization. max_iter is increased to ensure convergence.
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000))
])

# 2. Decision Tree Pipeline
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Fit both models on the training set ONLY
print("Training Logistic Regression pipeline...")
log_reg_pipeline.fit(X_train, y_train)

print("Training Decision Tree pipeline...")
tree_pipeline.fit(X_train, y_train)

print("Both models trained successfully on the training data!")


## Task 3: Evaluate on Hold-Out Test (Multiple Metrics)

**Error Analysis Interpretation:**
*Both models evaluated on the Hold-Out test set show that **False Negatives (FN) are significantly more common than False Positives (FP)** (for Logistic Regression, 950 FNs vs 494 FPs).*

*Because our primary goal is to **maximize Precision** (to avoid wasted outreach), having fewer False Positives is actually a fantastic result for our business case! Our Logistic Regression model achieves a Precision of ~73.8%, meaning when we contact someone, they are highly likely to actually be a high-earner. However, the high number of False Negatives means our Recall is lower (~59%), meaning we are missing out on identifying a lot of true high-earners. Since marketing budgets are limited, this is a trade-off we are generally happy to accept.*


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

# 1. Generate Predictions on Hold-Out Test
y_pred_log = log_reg_pipeline.predict(X_test)
y_prob_log = log_reg_pipeline.predict_proba(X_test)[:, 1]

y_pred_tree = tree_pipeline.predict(X_test)
y_prob_tree = tree_pipeline.predict_proba(X_test)[:, 1]

# Re-create Day 1 Baseline for comparison
if 'education-num' in X_test.columns:
    y_pred_rule = (X_test['education-num'] >= 13).astype(int)
else:
    higher_edu = ['Bachelors', 'Masters', 'Prof-school', 'Doctorate']
    y_pred_rule = X_test['education'].isin(higher_edu).astype(int)

# 2. Build Comparison Table
def evaluate_model(y_true, y_pred, y_prob, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC': roc_auc_score(y_true, y_prob),
        'PR AUC': average_precision_score(y_true, y_prob)
    }

metrics_rule = evaluate_model(y_test, y_pred_rule, y_pred_rule, "Day 1 Rule Baseline")
metrics_log = evaluate_model(y_test, y_pred_log, y_prob_log, "Logistic Regression")
metrics_tree = evaluate_model(y_test, y_pred_tree, y_prob_tree, "Decision Tree")

results_df = pd.DataFrame([metrics_rule, metrics_log, metrics_tree]).set_index('Model')
display(results_df)

# 3. Plot ROC and PR Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curves
RocCurveDisplay.from_estimator(log_reg_pipeline, X_test, y_test, ax=ax1, name='Logistic Regression')
RocCurveDisplay.from_estimator(tree_pipeline, X_test, y_test, ax=ax1, name='Decision Tree')
ax1.plot([0, 1], [0, 1], color='black', linestyle='--')
ax1.set_title('ROC Curves')

# PR Curves
PrecisionRecallDisplay.from_estimator(log_reg_pipeline, X_test, y_test, ax=ax2, name='Logistic Regression')
PrecisionRecallDisplay.from_estimator(tree_pipeline, X_test, y_test, ax=ax2, name='Decision Tree')
ax2.set_title('Precision-Recall Curves')

plt.tight_layout()
plt.show()

# 4. Confusion Matrices
print("--- Confusion Matrices ---")
print("\nLogistic Regression:")
print(confusion_matrix(y_test, y_pred_log))

print("\nDecision Tree:")
print(confusion_matrix(y_test, y_pred_tree))


## Task 4: Interpretability Check

### Logistic Regression Interpretation:
*   **Top Positive Indicators:** Features like high `capital-gain` (wealth), being married (`Married-civ-spouse`), holding `Exec-managerial` occupations, and higher `education-num` heavily increase the likelihood of earning >50K. 
*   **Top Negative Indicators:** Working in lower-wage jobs (`Priv-house-serv`, `Other-service`), being `Never-married`, and being `Female` in this 1994 census heavily decrease the likelihood of earning >50K. 

### Decision Tree Interpretation:
*   **Overfitting Check:** The tree has a massive depth of **52**! The Training Accuracy is **97.2%**, but the Test Accuracy drops to **82.1%**. This is a textbook example of **severe overfitting**. The tree essentially memorized the training data by splitting until every leaf was pure, which fails to generalize to unseen data.
*   **Top 3 Splits Logic:** The tree splits first on `marital-status_Married-civ-spouse`, then on `capital-gain`, and then on `education-num`. This logic is extremely sensible, as these perfectly align with the top coefficients found by our Logistic Regression model!


In [ ]:
import pandas as pd
from sklearn.tree import export_text

# --- Logistic Regression Interpretability ---
print("--- Logistic Regression Coefficients ---")
# Extract feature names from ColumnTransformer
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features = cat_encoder.get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_features)

# Map coefficients to names
coeffs = log_reg_pipeline.named_steps['classifier'].coef_[0]
coeff_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coeffs})

print("\nTop 10 Positive Coefficients (Increase chance of >50K):")
display(coeff_df.sort_values(by='Coefficient', ascending=False).head(10))

print("\nTop 10 Negative Coefficients (Decrease chance of >50K):")
display(coeff_df.sort_values(by='Coefficient', ascending=True).head(10))


# --- Decision Tree Interpretability ---
print("\n--- Decision Tree Overfitting Check ---")
dt_model = tree_pipeline.named_steps['classifier']
print(f"Tree Depth: {dt_model.get_depth()}")
print(f"Train Accuracy: {tree_pipeline.score(X_train, y_train):.3f}")
print(f"Test Accuracy:  {tree_pipeline.score(X_test, y_test):.3f}")

print("\n--- Top 3 Decision Tree Splits ---")
tree_rules = export_text(dt_model, feature_names=feature_names, max_depth=2)
print(tree_rules)


## Task 5: Write-Up & Model Selection for Day 3

**Model Selection Justification:**
Going into Day 3, we will continue developing **both** the Logistic Regression and Tree-based model families, but with different ultimate goals. The **Logistic Regression** model proved to be an incredibly strong baseline, achieving an F1-Score of ~0.66 and an excellent Precision of ~74%. Because it scales perfectly and offers high interpretability, it will serve as our primary production candidate if we decide we need a fast, highly-precise linear model. 

However, the raw **Decision Tree** severely overfit the training data. Despite this flaw, tree-based models naturally handle non-linear relationships and feature interactions (like age combined with specific occupations) much better than linear models. Therefore, we will upgrade the Decision Tree into a **Random Forest** or **Gradient Boosting Classifier** (like XGBoost) tomorrow. By using an ensemble approach to control the overfitting, we expect the tree-based models to ultimately surpass the Logistic Regression model in our primary F1-Score metric.

**Preprocessing Changes to Test Tomorrow:**
1.  **Log-Transforming Capital Gain/Loss:** We noticed these features dominate the top splits and coefficients, but they are extremely skewed (mostly zeros). Adding a `FunctionTransformer(np.log1p)` might stabilize them.
2.  **Binning Native Country:** Shrinking the high-cardinality `native-country` column into "US" vs "Non-US" to reduce noise and sparsity in the OneHotEncoder.


In [ ]:
import joblib
import os

# Save the preprocessing pipeline for reuse on Day 3
os.makedirs('models', exist_ok=True)
joblib.dump(preprocessor, 'models/preprocessor_day2.pkl')

print("Preprocessing pipeline successfully saved to 'models/preprocessor_day2.pkl' for reuse tomorrow!")
